In [ ]:
import ee

PROJECT_ID = "turkiye-border-wellbeing-2026"

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print("Earth Engine ready.")

Earth Engine ready.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
ROOT = Path("/content/drive/MyDrive/turkiye-border-wellbeing")

RAW_BOUNDARIES = ROOT / "data/raw/boundaries"
RAW_BOUNDARIES.mkdir(parents=True, exist_ok=True)

print(RAW_BOUNDARIES)

/content/drive/MyDrive/turkiye-border-wellbeing/data/raw/boundaries


In [ ]:
adm1 = ee.FeatureCollection("FAO/GAUL/2015/level1")

In [ ]:
turkiye_adm1 = adm1.filter(
    ee.Filter.eq("ADM0_NAME", "Turkey")
)

print("Province count:",
      turkiye_adm1.size().getInfo())

Province count: 80


In [ ]:
study_provinces = turkiye_adm1.filter(
    ee.Filter.inList(
        "ADM1_NAME",
        ["Edirne", "Kirklareli", "Tekirdag"]
    )
)

print("Selected provinces:",
      study_provinces.aggregate_array("ADM1_NAME").getInfo())

Selected provinces: ['Edirne', 'Kirklareli', 'Tekirdag']


In [ ]:
province_names = turkiye_adm1.aggregate_array("ADM1_NAME").getInfo()
province_names

['Adana',
 'Adiyaman',
 'Afyon',
 'Agri',
 'Aksaray',
 'Amasya',
 'Ankara',
 'Antalya',
 'Ardahan',
 'Artvin',
 'Aydin',
 'Balikesir',
 'Bartin',
 'Batman',
 'Bayburt',
 'Bilecik',
 'Bingol',
 'Bitlis',
 'Bolu',
 'Burdur',
 'Bursa',
 'Canakkale',
 'Cankiri',
 'Corum',
 'Denizli',
 'Diyarbakir',
 'Edirne',
 'Elazig',
 'Erzincan',
 'Erzurum',
 'Eskisehir',
 'Gaziantep',
 'Giresun',
 'Gumushane',
 'Hakkari',
 'Hatay',
 'Icel',
 'Igdir',
 'Isparta',
 'Istanbul',
 'Izmir',
 'K.maras',
 'Karabuk',
 'Karaman',
 'Kars',
 'Kastamonu',
 'Kayseri',
 'Kilis',
 'Kirikkale',
 'Kirklareli',
 'Kirsehir',
 'Kocaeli',
 'Konya',
 'Kutahya',
 'Malatya',
 'Manisa',
 'Mardin',
 'Mugla',
 'Mus',
 'Nevsehir',
 'Nigde',
 'Ordu',
 'Osmaniye',
 'Rize',
 'Sakarya',
 'Samsun',
 'Sanliurfa',
 'Siirt',
 'Sinop',
 'Sirnak',
 'Sivas',
 'Tekirdag',
 'Tokat',
 'Trabzon',
 'Tunceli',
 'Usak',
 'Van',
 'Yalova',
 'Yozgat',
 'Zonguldak']

In [ ]:
out_geojson = RAW_BOUNDARIES / "study_area_adm1_raw.geojson"

geemap.ee_to_geojson(
    study_provinces,
    filename=str(out_geojson)
)

print("Saved:", out_geojson)

Saved: /content/drive/MyDrive/turkiye-border-wellbeing/data/raw/boundaries/study_area_adm1_raw.geojson


In [ ]:
gdf = gpd.read_file(out_geojson)

print(gdf[["ADM0_NAME", "ADM1_NAME"]])
print("CRS:", gdf.crs)
print("Geometry valid:")
print(gdf.geometry.is_valid.value_counts())

  ADM0_NAME   ADM1_NAME
0    Turkey      Edirne
1    Turkey  Kirklareli
2    Turkey    Tekirdag
CRS: EPSG:4326
Geometry valid:
True    3
Name: count, dtype: int64


In [ ]:
out_gpkg = RAW_BOUNDARIES / "study_area_adm1_raw.gpkg"

gdf.to_file(
    out_gpkg,
    layer="study_area_adm1",
    driver="GPKG"
)

print("Saved:", out_gpkg)

Saved: /content/drive/MyDrive/turkiye-border-wellbeing/data/raw/boundaries/study_area_adm1_raw.gpkg


In [ ]:
adm2 = ee.FeatureCollection("FAO/GAUL/2015/level2")

study_adm2 = (
    adm2
    .filter(ee.Filter.eq("ADM0_NAME", "Turkey"))
    .filter(
        ee.Filter.inList(
            "ADM1_NAME",
            ["Edirne", "Kirklareli", "Tekirdag"]
        )
    )
)

print("District count:", study_adm2.size().getInfo())
print(
    study_adm2.aggregate_array("ADM2_NAME").getInfo()
)

District count: 26
['Enez', 'Havsa', 'Ipsala', 'Kesan', 'Meric', 'Merkez', 'Suleoglu', 'Uzunkopru', 'Babaeski', 'Luleburgaz', 'Merkez', 'Pehlivankoy', 'Pinarhisar', 'Vize', 'Cerkezkoy', 'Corlu', 'Hayrabolu', 'Malkara', 'Marmara-ereglisi', 'Merkez', 'Muratli', 'Saray', 'Sarkoy', 'Lalapasa', 'Demirkoy', 'Kofcaz']


In [ ]:
out_adm2_geojson = RAW_BOUNDARIES / "study_area_adm2_raw.geojson"

geemap.ee_to_geojson(
    study_adm2,
    filename=str(out_adm2_geojson)
)

print("Saved:", out_adm2_geojson)

Saved: /content/drive/MyDrive/turkiye-border-wellbeing/data/raw/boundaries/study_area_adm2_raw.geojson


In [ ]:
adm2_gdf = gpd.read_file(out_adm2_geojson)

adm2_gdf["district_id"] = (
    adm2_gdf["ADM1_NAME"].astype(str)
    + "_"
    + adm2_gdf["ADM2_NAME"].astype(str)
)

print(
    adm2_gdf[
        ["ADM1_NAME", "ADM2_NAME", "district_id"]
    ].sort_values(["ADM1_NAME", "ADM2_NAME"])
)

print("CRS:", adm2_gdf.crs)
print("Geometry valid:")
print(adm2_gdf.geometry.is_valid.value_counts())

     ADM1_NAME         ADM2_NAME                district_id
0       Edirne              Enez                Edirne_Enez
1       Edirne             Havsa               Edirne_Havsa
2       Edirne            Ipsala              Edirne_Ipsala
3       Edirne             Kesan               Edirne_Kesan
23      Edirne          Lalapasa            Edirne_Lalapasa
4       Edirne             Meric               Edirne_Meric
5       Edirne            Merkez              Edirne_Merkez
6       Edirne          Suleoglu            Edirne_Suleoglu
7       Edirne         Uzunkopru           Edirne_Uzunkopru
8   Kirklareli          Babaeski        Kirklareli_Babaeski
24  Kirklareli          Demirkoy        Kirklareli_Demirkoy
25  Kirklareli            Kofcaz          Kirklareli_Kofcaz
9   Kirklareli        Luleburgaz      Kirklareli_Luleburgaz
10  Kirklareli            Merkez          Kirklareli_Merkez
11  Kirklareli       Pehlivankoy     Kirklareli_Pehlivankoy
12  Kirklareli        Pinarhisar      Ki

In [ ]:
out_adm2_gpkg = RAW_BOUNDARIES / "study_area_adm2_raw.gpkg"

adm2_gdf.to_file(
    out_adm2_gpkg,
    layer="study_area_adm2",
    driver="GPKG"
)

print("Saved:", out_adm2_gpkg)

Saved: /content/drive/MyDrive/turkiye-border-wellbeing/data/raw/boundaries/study_area_adm2_raw.gpkg


In [ ]:
m = geemap.Map()

m.centerObject(study_provinces, 8)

m.addLayer(
    study_provinces,
    {},
    "ADM1 Study Area"
)

m.addLayer(
    study_adm2,
    {},
    "ADM2 Districts"
)

m

Map(center=[41.348163347949836, 27.16399346682171], controls=(WidgetControl(options=['position', 'transparent_…